# MODULE 2 — Dataset Discovery and Audit

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

**Datasets**
- BCI Competition IV Dataset 2a
- PhysioNet EEG Motor Movement/Imagery Dataset (EEGMMIDB)

**Purpose**

This module discovers and audits the actual files present on the MacBook before any
label harmonization, channel harmonization, preprocessing, or model training.

It does **not** make scientific decisions about the final class mapping yet.

The module records, where available:

- file paths and formats
- subjects
- sessions/runs
- recording IDs
- channel counts and names
- sampling rates
- durations
- annotations/events
- event counts
- NaN/Inf indicators
- corrupted/unreadable recordings
- duplicate candidates
- class/annotation labels as observed in the raw files
- dataset-level summaries
- a reproducible manifest

**Important:** This is an audit, not a preprocessing module.

## Scientific purpose

The cross-dataset experiment depends on facts about the local files.

Published dataset descriptions are useful, but the local files may contain:
- renamed recordings,
- missing runs,
- extra files,
- different folder layouts,
- duplicate recordings,
- incomplete annotations,
- channel naming differences,
- or corrupted files.

Therefore no later module is allowed to assume the local structure from memory.

Module 2 establishes the empirical dataset manifest that later modules must use.

## Cell 1 — Imports and project configuration

In [2]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION
# ============================================================

from __future__ import annotations

import os
import re
import json
import math
import hashlib
import platform
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.io as sio
import mne

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# Reuse the paths created in Module 1.
# ------------------------------------------------------------

PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")
BCI2A_ROOT = PROJECT_ROOT / "BCI IV-2a"
EEGMMIDB_ROOT = PROJECT_ROOT / "eegmmidb"

OUTPUT_ROOT = PROJECT_ROOT / "cross_dataset_mi_project"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"
LOG_ROOT = OUTPUT_ROOT / "logs"

MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

# Configuration for discovery.
SUPPORTED_SUFFIXES = {
    ".gdf",
    ".edf",
    ".bdf",
    ".fif",
    ".set",
    ".vhdr",
    ".eeg",
    ".vmrk",
    ".mat",
    ".txt",
    ".csv",
}

RAW_EEG_SUFFIXES = {
    ".gdf",
    ".edf",
    ".bdf",
    ".fif",
    ".set",
    ".vhdr",
}

# Full hashing is scientifically safer for duplicate detection but may take time.
ENABLE_FULL_SHA256 = True

# For raw-signal integrity checks, inspect a small deterministic sample rather
# than forcing every large recording to preload into memory.
ENABLE_SIGNAL_SAMPLE_CHECK = True
SIGNAL_SAMPLE_SECONDS = 20.0

print("=" * 78)
print("MODULE 2 — DATASET DISCOVERY AND AUDIT")
print("=" * 78)

print("BCI-IV-2a root :", BCI2A_ROOT)
print("EEGMMIDB root  :", EEGMMIDB_ROOT)

print("\nRoot existence:")
print("  BCI-IV-2a :", BCI2A_ROOT.exists())
print("  EEGMMIDB  :", EEGMMIDB_ROOT.exists())

if not BCI2A_ROOT.exists():
    print("\nWARNING: BCI-IV-2a directory is missing.")
if not EEGMMIDB_ROOT.exists():
    print("\nWARNING: EEGMMIDB directory is missing.")

MODULE 2 — DATASET DISCOVERY AND AUDIT
BCI-IV-2a root : /Users/ashokvarmabevara/Project2/BCI IV-2a
EEGMMIDB root  : /Users/ashokvarmabevara/Project2/eegmmidb

Root existence:
  BCI-IV-2a : True
  EEGMMIDB  : True


## Cell 2 — Recursive file discovery

This cell does not assume a folder structure.

The discovery table records:
- absolute path
- relative path
- suffix
- filename
- size
- modification time

The dataset-specific subject/session parsing is performed later and kept separate
from generic file discovery.

In [3]:
# ============================================================
# CELL 2 — RECURSIVE FILE DISCOVERY
# ============================================================

def discover_files(root: Path, dataset_name: str) -> pd.DataFrame:
    rows = []

    if not root.exists():
        return pd.DataFrame(columns=[
            "dataset",
            "absolute_path",
            "relative_path",
            "filename",
            "suffix",
            "size_bytes",
            "modified_time",
        ])

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix not in SUPPORTED_SUFFIXES:
            continue

        try:
            stat = path.stat()
            rows.append({
                "dataset": dataset_name,
                "absolute_path": str(path.resolve()),
                "relative_path": str(path.relative_to(root)),
                "filename": path.name,
                "suffix": suffix,
                "size_bytes": int(stat.st_size),
                "modified_time": pd.Timestamp(stat.st_mtime, unit="s"),
            })
        except OSError as exc:
            rows.append({
                "dataset": dataset_name,
                "absolute_path": str(path),
                "relative_path": str(path),
                "filename": path.name,
                "suffix": suffix,
                "size_bytes": np.nan,
                "modified_time": pd.NaT,
            })

    return (
        pd.DataFrame(rows)
        .sort_values(["dataset", "relative_path"])
        .reset_index(drop=True)
    )

files_bci = discover_files(BCI2A_ROOT, "BCI-IV-2a")
files_physionet = discover_files(EEGMMIDB_ROOT, "EEGMMIDB")

files_df = pd.concat([files_bci, files_physionet], ignore_index=True)

print("=" * 78)
print("DISCOVERY SUMMARY")
print("=" * 78)

print("Total discovered files:", len(files_df))

if len(files_df):
    print("\nBy dataset:")
    print(files_df.groupby("dataset").size().to_string())

    print("\nBy dataset and extension:")
    print(
        files_df.groupby(["dataset", "suffix"])
        .size()
        .sort_index()
        .to_string()
    )

print("\nFirst 20 discovered files:")
display(files_df.head(20))

DISCOVERY SUMMARY
Total discovered files: 1574

By dataset:
dataset
BCI-IV-2a      47
EEGMMIDB     1527

By dataset and extension:
dataset    suffix
BCI-IV-2a  .csv         2
           .gdf        18
           .mat        27
EEGMMIDB   .edf      1526
           .txt         1

First 20 discovered files:


,dataset,absolute_path,relative_path,filename,suffix,size_bytes,modified_time
0,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,A01E.gdf,A01E.gdf,.gdf,34363804,2026-03-02 04:00:41.066207885
1,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,A01T.gdf,A01T.gdf,.gdf,33640300,2026-04-10 02:09:06.250695944
2,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A01...,A01T.mat,A01T.mat,.mat,42806453,2026-02-25 10:15:53.666719913
3,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A02...,A02E.gdf,A02E.gdf,.gdf,33147080,2026-03-02 04:00:42.094147444
4,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A02...,A02T.gdf,A02T.gdf,.gdf,33872386,2026-04-10 02:09:09.064061403
5,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A02...,A02T.mat,A02T.mat,.mat,43068077,2026-02-25 10:15:57.372143984
6,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A03...,A03E.gdf,A03E.gdf,.gdf,32452650,2026-03-02 04:00:42.488198996
7,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A03...,A03T.gdf,A03T.gdf,.gdf,33040436,2026-04-10 02:09:09.733943939
8,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A03...,A03T.mat,A03T.mat,.mat,44057065,2026-02-25 10:15:58.252086878
9,BCI-IV-2a,/Users/ashokvarmabevara/Project2/BCI IV-2a/A04...,A04E.gdf,A04E.gdf,.gdf,33016790,2026-03-02 04:00:43.080705881


## Cell 3 — Dataset-specific subject / run parsing

Filename conventions differ between BCI-IV-2a and EEGMMIDB.

The parser is intentionally conservative:
- it records a parsed subject/run when confidently detected;
- otherwise it leaves the field as `UNKNOWN`;
- it does not silently infer experimental labels.

For BCI-IV-2a, common names such as `A01T.gdf` are recognized.

For EEGMMIDB, filenames such as `S001R04.edf` are parsed using the standard
subject/run pattern.

The actual discovered names remain the source of truth.

In [4]:
# ============================================================
# CELL 3 — SUBJECT / RUN PARSING
# ============================================================

def parse_bci2a_filename(filename: str) -> Dict[str, Any]:
    """
    Parse common BCI-IV-2a recording names such as A01T.gdf / A01E.gdf.
    """
    stem = Path(filename).stem.upper()

    # Common BCI IV-2a pattern: A01T, A01E
    match = re.search(r"A(\d{2})([TE])$", stem)

    if match:
        subject_num = int(match.group(1))
        session_code = match.group(2)

        return {
            "subject": f"S{subject_num:02d}",
            "session": session_code,
            "run": session_code,
            "recording_id": stem,
            "parse_status": "parsed",
        }

    # Broader fallback
    match = re.search(r"A(\d{2})", stem)
    if match:
        subject_num = int(match.group(1))
        return {
            "subject": f"S{subject_num:02d}",
            "session": "UNKNOWN",
            "run": "UNKNOWN",
            "recording_id": stem,
            "parse_status": "partial",
        }

    return {
        "subject": "UNKNOWN",
        "session": "UNKNOWN",
        "run": "UNKNOWN",
        "recording_id": stem,
        "parse_status": "unknown",
    }


def parse_physionet_filename(filename: str) -> Dict[str, Any]:
    """
    Parse standard EEGMMIDB names such as S001R01.edf.
    """
    stem = Path(filename).stem.upper()

    match = re.search(r"S(\d{3})R(\d{2})$", stem)

    if match:
        subject_num = int(match.group(1))
        run_num = int(match.group(2))

        return {
            "subject": f"S{subject_num:03d}",
            "session": "EEGMMIDB",
            "run": f"R{run_num:02d}",
            "recording_id": stem,
            "parse_status": "parsed",
        }

    return {
        "subject": "UNKNOWN",
        "session": "UNKNOWN",
        "run": "UNKNOWN",
        "recording_id": stem,
        "parse_status": "unknown",
    }


def parse_identity(row: pd.Series) -> Dict[str, Any]:
    if row["dataset"] == "BCI-IV-2a":
        return parse_bci2a_filename(row["filename"])

    if row["dataset"] == "EEGMMIDB":
        return parse_physionet_filename(row["filename"])

    return {
        "subject": "UNKNOWN",
        "session": "UNKNOWN",
        "run": "UNKNOWN",
        "recording_id": Path(row["filename"]).stem,
        "parse_status": "unknown",
    }


identity_rows = [parse_identity(row) for _, row in files_df.iterrows()]
identity_df = pd.DataFrame(identity_rows)

files_df = pd.concat(
    [files_df.reset_index(drop=True), identity_df.reset_index(drop=True)],
    axis=1,
)

print("=" * 78)
print("IDENTITY PARSING")
print("=" * 78)

print(
    files_df.groupby(
        ["dataset", "parse_status"],
        dropna=False
    ).size().to_string()
)

display(
    files_df[
        [
            "dataset",
            "filename",
            "subject",
            "session",
            "run",
            "parse_status",
        ]
    ].head(30)
)

IDENTITY PARSING
dataset    parse_status
BCI-IV-2a  parsed            45
           unknown            2
EEGMMIDB   parsed          1526
           unknown            1


,dataset,filename,subject,session,run,parse_status
0,BCI-IV-2a,A01E.gdf,S01,E,E,parsed
1,BCI-IV-2a,A01T.gdf,S01,T,T,parsed
2,BCI-IV-2a,A01T.mat,S01,T,T,parsed
3,BCI-IV-2a,A02E.gdf,S02,E,E,parsed
4,BCI-IV-2a,A02T.gdf,S02,T,T,parsed
5,BCI-IV-2a,A02T.mat,S02,T,T,parsed
6,BCI-IV-2a,A03E.gdf,S03,E,E,parsed
7,BCI-IV-2a,A03T.gdf,S03,T,T,parsed
8,BCI-IV-2a,A03T.mat,S03,T,T,parsed
9,BCI-IV-2a,A04E.gdf,S04,E,E,parsed


## Cell 4 — File hashes and duplicate candidates

Duplicate detection is performed at the file level.

Two mechanisms are recorded:
1. identical file size;
2. optional full SHA-256 hash.

A hash match is strong evidence of byte-identical files.

This does not by itself prove that two recordings are scientifically duplicate
trials if the files are independently encoded differently, so duplicate analysis
remains an audit signal rather than a preprocessing decision.

In [6]:
# ============================================================
# CELL 4 — HASHING / DUPLICATE DETECTION (FIXED)
# ============================================================

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> Optional[str]:
    """Compute SHA-256 without loading the entire file into RAM."""
    try:
        h = hashlib.sha256()

        with open(path, "rb") as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                h.update(chunk)

        return h.hexdigest()

    except Exception:
        return None


print("Computing file-level duplicate metadata...")

hash_rows = []

for _, row in tqdm(
    files_df.iterrows(),
    total=len(files_df),
    desc="Hashing files",
):
    path = Path(row["absolute_path"])

    sha = sha256_file(path) if ENABLE_FULL_SHA256 else None

    hash_rows.append({
        "absolute_path": str(path.resolve()),
        "sha256": sha,
    })

hash_df = pd.DataFrame(hash_rows)

# ------------------------------------------------------------
# Defensive validation
# ------------------------------------------------------------

assert "absolute_path" in files_df.columns
assert "absolute_path" in hash_df.columns

if files_df["absolute_path"].duplicated().any():
    duplicate_paths = files_df.loc[
        files_df["absolute_path"].duplicated(keep=False),
        "absolute_path"
    ].unique().tolist()

    raise RuntimeError(
        "Duplicate absolute paths found in files_df: "
        f"{duplicate_paths[:5]}"
    )

if hash_df["absolute_path"].duplicated().any():
    duplicate_paths = hash_df.loc[
        hash_df["absolute_path"].duplicated(keep=False),
        "absolute_path"
    ].unique().tolist()

    raise RuntimeError(
        "Duplicate absolute paths found in hash_df: "
        f"{duplicate_paths[:5]}"
    )

# Remove old hash column in case this cell is re-run.
files_df = files_df.drop(columns=["sha256"], errors="ignore")
files_df = files_df.drop(columns=["duplicate_hash_group"], errors="ignore")

# ------------------------------------------------------------
# FIX:
# Join ONLY on absolute_path.
# ------------------------------------------------------------

files_df = files_df.merge(
    hash_df,
    on="absolute_path",
    how="left",
    validate="one_to_one",
)

# ------------------------------------------------------------
# Duplicate detection
# ------------------------------------------------------------

files_df["duplicate_hash_group"] = (
    files_df["sha256"]
    .replace("", np.nan)
)

duplicate_hash_groups = (
    files_df.dropna(subset=["duplicate_hash_group"])
    .groupby("duplicate_hash_group")
    .filter(lambda g: len(g) > 1)
    .sort_values("duplicate_hash_group")
)

print("\nHashing complete.")
print("Files discovered :", len(files_df))
print("Files hashed     :", len(hash_df))
print(
    "Successful hashes:",
    int(hash_df["sha256"].notna().sum())
)
print(
    "Failed hashes    :",
    int(hash_df["sha256"].isna().sum())
)

if len(duplicate_hash_groups):
    print("\nPotential byte-identical duplicate files:")
    display(
        duplicate_hash_groups[
            [
                "dataset",
                "relative_path",
                "subject",
                "run",
                "size_bytes",
                "sha256",
            ]
        ]
    )
else:
    print("\nNo byte-identical duplicate files detected.")

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(files_df) == len(hash_df)

assert (
    files_df["sha256"].notna().sum()
    == hash_df["sha256"].notna().sum()
)

print("\nCell 4 validation: PASS")

Computing file-level duplicate metadata...


Hashing files:   0%|          | 0/1574 [00:00<?, ?it/s]


Hashing complete.
Files discovered : 1574
Files hashed     : 1574
Successful hashes: 1574
Failed hashes    : 0

No byte-identical duplicate files detected.

Cell 4 validation: PASS


## Cell 5 — Generic EEG header / annotation inspection

This is the main audit pass.

Supported raw EEG formats are inspected using MNE without preloading the entire
recording whenever possible.

For each recording the audit attempts to collect:
- sampling rate
- number of channels
- channel names
- recording duration
- annotation count
- annotation descriptions
- whether the file can be read
- whether the recording contains NaN/Inf values in a deterministic sample

No preprocessing is performed here.

In [7]:
# ============================================================
# CELL 5 — RAW EEG HEADER / ANNOTATION AUDIT
# ============================================================

RAW_READERS = {
    ".gdf": mne.io.read_raw_gdf,
    ".edf": mne.io.read_raw_edf,
    ".bdf": mne.io.read_raw_bdf,
    ".fif": mne.io.read_raw_fif,
    ".set": mne.io.read_raw_eeglab,
    ".vhdr": mne.io.read_raw_brainvision,
}


def _safe_float(value) -> Optional[float]:
    try:
        return float(value)
    except Exception:
        return None


def summarize_annotation_descriptions(raw) -> Dict[str, int]:
    counter = Counter()

    try:
        descriptions = list(raw.annotations.description)
        for desc in descriptions:
            counter[str(desc)] += 1
    except Exception:
        pass

    return dict(counter)


def signal_sample_quality(raw, sample_seconds: float = SIGNAL_SAMPLE_SECONDS):
    """
    Deterministic low-cost signal quality check.

    Reads only a bounded beginning segment. This is NOT preprocessing and is
    not used to compute normalization statistics.
    """
    if not ENABLE_SIGNAL_SAMPLE_CHECK:
        return {
            "sample_checked": False,
            "sample_nan": None,
            "sample_inf": None,
            "sample_max_abs": None,
            "sample_ptp_median": None,
            "sample_error": None,
        }

    try:
        sfreq = float(raw.info["sfreq"])
        n_times = int(raw.n_times)

        n_sample = int(min(
            n_times,
            max(1, round(sample_seconds * sfreq))
        ))

        # preload=False Raw objects can still load a selected sample region.
        data = raw.get_data(
            start=0,
            stop=n_sample,
            reject_by_annotation="omit",
        )

        finite_mask = np.isfinite(data)

        if finite_mask.any():
            finite_values = data[finite_mask]
            max_abs = float(np.max(np.abs(finite_values)))
        else:
            max_abs = None

        ptp = np.ptp(data, axis=1)
        ptp_median = float(np.nanmedian(ptp))

        return {
            "sample_checked": True,
            "sample_nan": bool(np.isnan(data).any()),
            "sample_inf": bool(np.isinf(data).any()),
            "sample_max_abs": max_abs,
            "sample_ptp_median": ptp_median,
            "sample_error": None,
        }

    except Exception as exc:
        return {
            "sample_checked": False,
            "sample_nan": None,
            "sample_inf": None,
            "sample_max_abs": None,
            "sample_ptp_median": None,
            "sample_error": repr(exc),
        }


def inspect_raw_file(path: Path) -> Dict[str, Any]:
    suffix = path.suffix.lower()

    result = {
        "file_readable": False,
        "read_error": None,
        "reader": None,
        "sfreq_hz": None,
        "n_channels": None,
        "n_times": None,
        "duration_sec": None,
        "channel_names": None,
        "channel_types": None,
        "annotation_count": None,
        "annotation_descriptions": None,
        "meas_date": None,
        "sampling_rates_raw": None,
        "sample_checked": False,
        "sample_nan": None,
        "sample_inf": None,
        "sample_max_abs": None,
        "sample_ptp_median": None,
        "sample_error": None,
    }

    if suffix not in RAW_READERS:
        result["read_error"] = f"Unsupported raw-reader suffix: {suffix}"
        return result

    reader = RAW_READERS[suffix]
    result["reader"] = getattr(reader, "__name__", str(reader))

    try:
        raw = reader(
            str(path),
            preload=False,
            verbose="ERROR",
        )

        result["file_readable"] = True
        result["sfreq_hz"] = _safe_float(raw.info.get("sfreq"))
        result["n_channels"] = int(raw.info.get("nchan", len(raw.ch_names)))
        result["n_times"] = int(raw.n_times)

        sfreq = result["sfreq_hz"]
        if sfreq and sfreq > 0:
            result["duration_sec"] = float(raw.n_times / sfreq)

        result["channel_names"] = list(raw.ch_names)

        try:
            result["channel_types"] = list(
                mne.io.pick.channel_type_vec(raw.info)
            )
        except Exception:
            result["channel_types"] = None

        try:
            result["annotation_count"] = int(len(raw.annotations))
        except Exception:
            result["annotation_count"] = 0

        result["annotation_descriptions"] = summarize_annotation_descriptions(raw)

        try:
            result["meas_date"] = str(raw.info.get("meas_date"))
        except Exception:
            result["meas_date"] = None

        try:
            result["sampling_rates_raw"] = np.unique(
                raw.info["sfreq"]
            ).tolist()
        except Exception:
            result["sampling_rates_raw"] = None

        quality = signal_sample_quality(raw)
        result.update(quality)

        # Explicitly close the underlying file where supported.
        try:
            raw.close()
        except Exception:
            pass

    except Exception as exc:
        result["read_error"] = repr(exc)

    return result


raw_audit_rows = []

raw_files_df = files_df[
    files_df["suffix"].isin(RAW_EEG_SUFFIXES)
].copy()

print(f"Raw EEG recordings detected: {len(raw_files_df)}")

for _, row in tqdm(
    raw_files_df.iterrows(),
    total=len(raw_files_df),
    desc="Inspecting EEG headers",
):
    path = Path(row["absolute_path"])

    inspection = inspect_raw_file(path)

    raw_audit_rows.append({
        **{
            "dataset": row["dataset"],
            "absolute_path": row["absolute_path"],
            "relative_path": row["relative_path"],
            "filename": row["filename"],
            "suffix": row["suffix"],
            "subject": row["subject"],
            "session": row["session"],
            "run": row["run"],
            "recording_id": row["recording_id"],
            "parse_status": row["parse_status"],
        },
        **inspection,
    })

raw_audit_df = pd.DataFrame(raw_audit_rows)

print("\nHeader audit complete.")

display(
    raw_audit_df[
        [
            "dataset",
            "filename",
            "subject",
            "run",
            "file_readable",
            "sfreq_hz",
            "n_channels",
            "n_times",
            "duration_sec",
            "annotation_count",
            "sample_nan",
            "sample_inf",
        ]
    ].head(30)
)

Raw EEG recordings detected: 1544


Inspecting EEG headers:   0%|          | 0/1544 [00:00<?, ?it/s]


Header audit complete.


,dataset,filename,subject,run,file_readable,sfreq_hz,n_channels,n_times,duration_sec,annotation_count,sample_nan,sample_inf
0,BCI-IV-2a,A01E.gdf,S01,E,True,250.0,25,687000,2748.000,595,False,False
1,BCI-IV-2a,A01T.gdf,S01,T,True,250.0,25,672528,2690.112,603,False,False
2,BCI-IV-2a,A02E.gdf,S02,E,True,250.0,25,662666,2650.664,593,False,False
3,BCI-IV-2a,A02T.gdf,S02,T,True,250.0,25,677169,2708.676,606,False,False
4,BCI-IV-2a,A03E.gdf,S03,E,True,250.0,25,648775,2595.100,603,False,False
5,BCI-IV-2a,A03T.gdf,S03,T,True,250.0,25,660530,2642.120,606,False,False
6,BCI-IV-2a,A04E.gdf,S04,E,True,250.0,25,660047,2640.188,648,False,False
7,BCI-IV-2a,A04T.gdf,S04,T,True,250.0,25,600915,2403.660,610,False,False
8,BCI-IV-2a,A05E.gdf,S05,E,True,250.0,25,679863,2719.452,600,False,False
9,BCI-IV-2a,A05T.gdf,S05,T,True,250.0,25,686120,2744.480,614,False,False


## Cell 6 — Channel inventory and montage inconsistencies

This cell creates:
1. one row per recording/channel;
2. one summary table of channel-count/name patterns by dataset;
3. a normalized channel-name inventory.

The notebook does not yet select a common montage.
That decision belongs to Module 4 after the actual inventories are verified.

In [8]:
# ============================================================
# CELL 6 — CHANNEL INVENTORY
# ============================================================

channel_rows = []

for _, row in raw_audit_df.iterrows():
    channel_names = row.get("channel_names")

    if not isinstance(channel_names, list):
        continue

    for idx, ch_name in enumerate(channel_names):
        channel_rows.append({
            "dataset": row["dataset"],
            "subject": row["subject"],
            "session": row["session"],
            "run": row["run"],
            "recording_id": row["recording_id"],
            "filename": row["filename"],
            "channel_index": idx,
            "channel_name_raw": ch_name,
            "channel_name_normalized": str(ch_name).strip().upper(),
        })

channel_inventory_df = pd.DataFrame(channel_rows)

print("=" * 78)
print("CHANNEL INVENTORY")
print("=" * 78)

if len(channel_inventory_df):
    channel_count_summary = (
        raw_audit_df.groupby(
            ["dataset", "n_channels"],
            dropna=False
        )
        .size()
        .reset_index(name="recordings")
        .sort_values(["dataset", "n_channels"])
    )

    print("\nChannel-count distribution:")
    display(channel_count_summary)

    print("\nMost frequent channel names by dataset:")
    for dataset_name, sub in channel_inventory_df.groupby("dataset"):
        counts = (
            sub.groupby("channel_name_normalized")
            .size()
            .sort_values(ascending=False)
            .head(80)
        )
        print(f"\n--- {dataset_name} ---")
        display(counts.to_frame("recording_channel_occurrences"))
else:
    print("No channel inventory available because no raw recordings were readable.")

CHANNEL INVENTORY

Channel-count distribution:


,dataset,n_channels,recordings
0,BCI-IV-2a,25,18
1,EEGMMIDB,64,1526



Most frequent channel names by dataset:

--- BCI-IV-2a ---


,recording_channel_occurrences
channel_name_normalized,
EEG-0,18
EEG-6,18
EOG-LEFT,18
EOG-CENTRAL,18
EEG-PZ,18
EEG-FZ,18
EEG-CZ,18
EEG-C4,18
EEG-C3,18



--- EEGMMIDB ---


,recording_channel_occurrences
channel_name_normalized,
AF3.,1526
AF4.,1526
FP1.,1526
FP2.,1526
FPZ.,1526
...,...
F8..,1526
FC1.,1526
FC2.,1526


## Cell 7 — Sampling-rate, duration, annotation and readability audit

This summarizes important recording-level properties.

Annotation descriptions are kept as **observed labels**, not interpreted yet.
That distinction is important: Module 3 will make the scientific label mapping.

In [9]:
# ============================================================
# CELL 7 — RECORDING PROPERTY SUMMARIES
# ============================================================

print("=" * 78)
print("RECORDING PROPERTY SUMMARIES")
print("=" * 78)

if len(raw_audit_df):
    print("\nSampling-rate distribution:")
    display(
        raw_audit_df.groupby(
            ["dataset", "sfreq_hz"],
            dropna=False
        ).size().reset_index(name="recordings")
        .sort_values(["dataset", "sfreq_hz"])
    )

    print("\nChannel-count distribution:")
    display(
        raw_audit_df.groupby(
            ["dataset", "n_channels"],
            dropna=False
        ).size().reset_index(name="recordings")
        .sort_values(["dataset", "n_channels"])
    )

    duration_summary = (
        raw_audit_df.groupby("dataset")["duration_sec"]
        .agg(["count", "min", "median", "mean", "max"])
        .reset_index()
    )

    print("\nDuration summary (seconds):")
    display(duration_summary)

    print("\nReadable / unreadable files:")
    readability = (
        raw_audit_df.groupby(
            ["dataset", "file_readable"],
            dropna=False
        ).size().reset_index(name="recordings")
    )
    display(readability)

    print("\nAnnotation counts:")
    annotation_summary = (
        raw_audit_df.groupby("dataset")["annotation_count"]
        .agg(["count", "min", "median", "mean", "max"])
        .reset_index()
    )
    display(annotation_summary)
else:
    print("No readable recording summaries available.")

RECORDING PROPERTY SUMMARIES

Sampling-rate distribution:


,dataset,sfreq_hz,recordings
0,BCI-IV-2a,250.0,18
1,EEGMMIDB,128.0,36
2,EEGMMIDB,160.0,1490



Channel-count distribution:


,dataset,n_channels,recordings
0,BCI-IV-2a,25,18
1,EEGMMIDB,64,1526



Duration summary (seconds):


,dataset,count,min,median,mean,max
0,BCI-IV-2a,18,2403.66,2696.852,2677.035556,2751.168
1,EEGMMIDB,1526,37.00,123.000,114.498034,181.000



Readable / unreadable files:


,dataset,file_readable,recordings
0,BCI-IV-2a,True,18
1,EEGMMIDB,True,1526



Annotation counts:


,dataset,count,min,median,mean,max
0,BCI-IV-2a,18,593,606.0,614.888889,661
1,EEGMMIDB,1526,1,30.0,25.929882,44


## Cell 8 — Annotation/event inventory

This cell preserves the event/annotation descriptions exactly as observed.

For each recording, the resulting table contains:
- dataset
- subject
- run
- annotation label
- count

No event label is assigned to left/right/feet/tongue yet.

In [10]:
# ============================================================
# CELL 8 — ANNOTATION / EVENT INVENTORY
# ============================================================

annotation_rows = []

for _, row in raw_audit_df.iterrows():
    desc_map = row.get("annotation_descriptions")

    if not isinstance(desc_map, dict):
        continue

    for desc, count in desc_map.items():
        annotation_rows.append({
            "dataset": row["dataset"],
            "subject": row["subject"],
            "session": row["session"],
            "run": row["run"],
            "recording_id": row["recording_id"],
            "filename": row["filename"],
            "annotation": str(desc),
            "count": int(count),
        })

annotation_df = pd.DataFrame(annotation_rows)

print("=" * 78)
print("OBSERVED ANNOTATION INVENTORY")
print("=" * 78)

if len(annotation_df):
    print("\nUnique annotations by dataset:")

    for dataset_name, sub in annotation_df.groupby("dataset"):
        summary = (
            sub.groupby("annotation")["count"]
            .sum()
            .sort_values(ascending=False)
            .to_frame("total_annotation_count")
        )

        print(f"\n--- {dataset_name} ---")
        display(summary.head(100))

    print("\nAnnotation counts by run example:")
    display(
        annotation_df
        .sort_values(
            ["dataset", "subject", "run", "annotation"]
        )
        .head(100)
    )
else:
    print("No annotations were found in the readable raw recordings.")

OBSERVED ANNOTATION INVENTORY

Unique annotations by dataset:

--- BCI-IV-2a ---


,total_annotation_count
annotation,
768,5184
783,2592
769,648
770,648
771,648
772,648
1023,488
32766,160
1072,18



--- EEGMMIDB ---


,total_annotation_count
annotation,
T0,19893
T1,9858
T2,9818



Annotation counts by run example:


,dataset,subject,session,run,recording_id,filename,annotation,count
6,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,1023,7
3,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,1072,1
1,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,276,1
2,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,277,1
0,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,32766,9
...,...,...,...,...,...,...,...,...
94,BCI-IV-2a,S06,T,T,A06T,A06T.gdf,768,288
97,BCI-IV-2a,S06,T,T,A06T,A06T.gdf,769,72
96,BCI-IV-2a,S06,T,T,A06T,A06T.gdf,770,72
98,BCI-IV-2a,S06,T,T,A06T,A06T.gdf,771,72


## Cell 9 — Corruption and signal-quality audit

The audit distinguishes:
- unreadable files;
- annotation/read errors;
- sampled NaN values;
- sampled Inf values;
- unusually large finite values;
- failed signal sampling.

A small deterministic sample is used so that Module 2 remains computationally
reasonable. This is not a substitute for full signal validation in Module 5.

In [11]:
# ============================================================
# CELL 9 — CORRUPTION / QUALITY AUDIT
# ============================================================

if len(raw_audit_df):
    quality_flags = raw_audit_df.copy()

    quality_flags["flag_unreadable"] = ~quality_flags["file_readable"].fillna(False)
    quality_flags["flag_sample_nan"] = quality_flags["sample_nan"].fillna(False)
    quality_flags["flag_sample_inf"] = quality_flags["sample_inf"].fillna(False)
    quality_flags["flag_sample_check_failed"] = (
        quality_flags["sample_checked"].fillna(False) == False
    )

    # This threshold is ONLY a diagnostic warning.
    # It is not used to reject data automatically.
    quality_flags["flag_large_amplitude_warning"] = (
        quality_flags["sample_max_abs"]
        .notna()
        & (quality_flags["sample_max_abs"] > 1e6)
    )

    quality_columns = [
        "dataset",
        "subject",
        "run",
        "filename",
        "file_readable",
        "sample_checked",
        "sample_nan",
        "sample_inf",
        "sample_max_abs",
        "sample_ptp_median",
        "read_error",
        "sample_error",
        "flag_unreadable",
        "flag_sample_nan",
        "flag_sample_inf",
        "flag_sample_check_failed",
        "flag_large_amplitude_warning",
    ]

    quality_problem_df = quality_flags[
        quality_flags[
            [
                "flag_unreadable",
                "flag_sample_nan",
                "flag_sample_inf",
                "flag_sample_check_failed",
                "flag_large_amplitude_warning",
            ]
        ].any(axis=1)
    ][quality_columns]

    print("=" * 78)
    print("QUALITY / CORRUPTION AUDIT")
    print("=" * 78)

    print("Files with at least one quality flag:", len(quality_problem_df))

    if len(quality_problem_df):
        display(quality_problem_df)
    else:
        print("No sampled signal-quality or readability problems detected.")

    print("\nQuality flag counts:")
    flag_summary = pd.DataFrame({
        "flag": [
            "unreadable",
            "sample_nan",
            "sample_inf",
            "sample_check_failed",
            "large_amplitude_warning",
        ],
        "count": [
            int(quality_flags["flag_unreadable"].sum()),
            int(quality_flags["flag_sample_nan"].sum()),
            int(quality_flags["flag_sample_inf"].sum()),
            int(quality_flags["flag_sample_check_failed"].sum()),
            int(quality_flags["flag_large_amplitude_warning"].sum()),
        ],
    })
    display(flag_summary)
else:
    quality_flags = pd.DataFrame()
    quality_problem_df = pd.DataFrame()
    print("No raw recordings available for quality audit.")

QUALITY / CORRUPTION AUDIT
Files with at least one quality flag: 0
No sampled signal-quality or readability problems detected.

Quality flag counts:


,flag,count
0,unreadable,0
1,sample_nan,0
2,sample_inf,0
3,sample_check_failed,0
4,large_amplitude_warning,0


## Cell 10 — MAT-file inspection for BCI-IV-2a sidecar data

BCI-IV-2a directories commonly contain MATLAB sidecar files in addition to the
GDF recordings.

This cell inspects MAT files structurally without treating their contents as
final labels.

The output reports:
- top-level keys
- MATLAB object/array shapes
- basic data types

It does not alter or overwrite any files.

In [12]:
# ============================================================
# CELL 10 — MATLAB SIDECAR AUDIT
# ============================================================

mat_rows = []

mat_files = files_df[
    (files_df["dataset"] == "BCI-IV-2a")
    & (files_df["suffix"] == ".mat")
].copy()

print(f"BCI-IV-2a MAT files: {len(mat_files)}")

for _, row in tqdm(
    mat_files.iterrows(),
    total=len(mat_files),
    desc="Inspecting MAT files",
):
    path = Path(row["absolute_path"])

    result = {
        "dataset": row["dataset"],
        "absolute_path": row["absolute_path"],
        "relative_path": row["relative_path"],
        "filename": row["filename"],
        "subject": row["subject"],
        "run": row["run"],
        "mat_readable": False,
        "mat_keys": None,
        "mat_summary": None,
        "mat_error": None,
    }

    try:
        mat = sio.loadmat(
            str(path),
            squeeze_me=False,
            struct_as_record=False,
        )

        public_keys = [k for k in mat.keys() if not k.startswith("__")]

        summary = {}

        for key in public_keys:
            value = mat[key]

            try:
                shape = tuple(value.shape)
            except Exception:
                shape = None

            summary[key] = {
                "type": str(type(value).__name__),
                "shape": shape,
                "dtype": str(getattr(value, "dtype", None)),
            }

        result["mat_readable"] = True
        result["mat_keys"] = public_keys
        result["mat_summary"] = summary

    except Exception as exc:
        result["mat_error"] = repr(exc)

    mat_rows.append(result)

mat_audit_df = pd.DataFrame(mat_rows)

if len(mat_audit_df):
    print("\nMAT readability:")
    display(
        mat_audit_df[
            [
                "filename",
                "subject",
                "run",
                "mat_readable",
                "mat_keys",
                "mat_error",
            ]
        ]
    )
else:
    print("No BCI-IV-2a MAT files were discovered.")

BCI-IV-2a MAT files: 27


Inspecting MAT files:   0%|          | 0/27 [00:00<?, ?it/s]


MAT readability:


,filename,subject,run,mat_readable,mat_keys,mat_error
0,A01T.mat,S01,T,True,[data],None
1,A02T.mat,S02,T,True,[data],None
2,A03T.mat,S03,T,True,[data],None
3,A04T.mat,S04,T,True,[data],None
4,A05T.mat,S05,T,True,[data],None
5,A06T.mat,S06,T,True,[data],None
6,A07T.mat,S07,T,True,[data],None
7,A08T.mat,S08,T,True,[data],None
8,A09T.mat,S09,T,True,[data],None
9,A01E.mat,S01,E,True,[classlabel],None


## Cell 11 — Subject and run manifest

This table aggregates recording-level information into the subject/run structure
that later modules will use.

The manifest deliberately keeps the original dataset identity and file paths.

No trials are yet created.

In [13]:
# ============================================================
# CELL 11 — SUBJECT / RUN MANIFEST
# ============================================================

manifest_columns = [
    "dataset",
    "subject",
    "session",
    "run",
    "recording_id",
    "filename",
    "relative_path",
    "absolute_path",
    "suffix",
    "file_readable",
    "sfreq_hz",
    "n_channels",
    "n_times",
    "duration_sec",
    "annotation_count",
    "annotation_descriptions",
    "parse_status",
    "sample_nan",
    "sample_inf",
    "sample_max_abs",
    "read_error",
]

manifest_df = raw_audit_df.copy()

for col in manifest_columns:
    if col not in manifest_df.columns:
        manifest_df[col] = np.nan

manifest_df = manifest_df[manifest_columns].copy()

print("=" * 78)
print("SUBJECT / RUN MANIFEST")
print("=" * 78)

if len(manifest_df):
    subject_counts = (
        manifest_df.groupby("dataset")["subject"]
        .nunique()
        .to_frame("unique_subjects")
        .reset_index()
    )

    print("\nUnique subjects:")
    display(subject_counts)

    print("\nSubjects by dataset:")
    for dataset_name, sub in manifest_df.groupby("dataset"):
        subjects = sorted(
            x for x in sub["subject"].dropna().unique()
            if str(x) != "UNKNOWN"
        )
        print(f"\n{dataset_name}: {len(subjects)} subjects")
        print(subjects)

    print("\nRun/recording counts by subject:")
    run_counts = (
        manifest_df.groupby(
            ["dataset", "subject"],
            dropna=False
        )
        .size()
        .reset_index(name="recordings")
        .sort_values(["dataset", "subject"])
    )
    display(run_counts)
else:
    print("No readable raw EEG recordings are available.")

SUBJECT / RUN MANIFEST

Unique subjects:


,dataset,unique_subjects
0,BCI-IV-2a,9
1,EEGMMIDB,109



Subjects by dataset:

BCI-IV-2a: 9 subjects
['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']

EEGMMIDB: 109 subjects
['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050', 'S051', 'S052', 'S053', 'S054', 'S055', 'S056', 'S057', 'S058', 'S059', 'S060', 'S061', 'S062', 'S063', 'S064', 'S065', 'S066', 'S067', 'S068', 'S069', 'S070', 'S071', 'S072', 'S073', 'S074', 'S075', 'S076', 'S077', 'S078', 'S079', 'S080', 'S081', 'S082', 'S083', 'S084', 'S085', 'S086', 'S087', 'S088', 'S089', 'S090', 'S091', 'S092', 'S093', 'S094', 'S095', 'S096', 'S097', 'S098', 'S099', 'S100', 'S101', 'S102', 'S103', 'S104', 'S105', 'S106', 'S107', 'S108', 'S

,dataset,subject,recordings
0,BCI-IV-2a,S01,2
1,BCI-IV-2a,S02,2
2,BCI-IV-2a,S03,2
3,BCI-IV-2a,S04,2
4,BCI-IV-2a,S05,2
...,...,...,...
113,EEGMMIDB,S105,14
114,EEGMMIDB,S106,14
115,EEGMMIDB,S107,14
116,EEGMMIDB,S108,14


## Cell 12 — Detect inconsistent sampling rates, channel counts and channel sets

These checks do not fix inconsistencies.

They identify exactly where later harmonization work is necessary.

In [14]:
# ============================================================
# CELL 12 — INCONSISTENCY DETECTION
# ============================================================

inconsistency_reports = {}

if len(raw_audit_df):
    # Sampling-rate inconsistencies
    sfreq_by_dataset = (
        raw_audit_df.groupby("dataset")["sfreq_hz"]
        .apply(lambda s: sorted(
            [float(x) for x in s.dropna().unique()]
        ))
        .to_dict()
    )

    # Channel-count inconsistencies
    nchan_by_dataset = (
        raw_audit_df.groupby("dataset")["n_channels"]
        .apply(lambda s: sorted(
            [int(x) for x in s.dropna().unique()]
        ))
        .to_dict()
    )

    # Exact ordered channel-set patterns
    def channel_signature(value):
        if isinstance(value, list):
            return tuple(str(x).strip().upper() for x in value)
        return None

    raw_audit_df["channel_signature"] = raw_audit_df["channel_names"].map(
        channel_signature
    )

    channel_pattern_counts = (
        raw_audit_df.groupby(
            ["dataset", "channel_signature"],
            dropna=False
        )
        .size()
        .reset_index(name="recordings")
        .sort_values(["dataset", "recordings"], ascending=[True, False])
    )

    inconsistency_reports = {
        "sampling_rates": sfreq_by_dataset,
        "channel_counts": nchan_by_dataset,
        "channel_patterns": channel_pattern_counts,
    }

    print("=" * 78)
    print("INCONSISTENCY CHECK")
    print("=" * 78)

    print("\nSampling rates by dataset:")
    for dataset_name, values in sfreq_by_dataset.items():
        print(f"  {dataset_name}: {values}")

    print("\nChannel counts by dataset:")
    for dataset_name, values in nchan_by_dataset.items():
        print(f"  {dataset_name}: {values}")

    print("\nDistinct ordered channel-set patterns:")
    display(channel_pattern_counts.head(30))
else:
    print("No raw recordings available for inconsistency analysis.")

INCONSISTENCY CHECK

Sampling rates by dataset:
  BCI-IV-2a: [250.0]
  EEGMMIDB: [128.0, 160.0]

Channel counts by dataset:
  BCI-IV-2a: [25]
  EEGMMIDB: [64]

Distinct ordered channel-set patterns:


,dataset,channel_signature,recordings
0,BCI-IV-2a,"(EEG-FZ, EEG-0, EEG-1, EEG-2, EEG-3, EEG-4, EE...",18
1,EEGMMIDB,"(FC5., FC3., FC1., FCZ., FC2., FC4., FC6., C5....",1526


## Cell 13 — Observed class/annotation overview

The audit should show what the local files actually contain before Module 3
decides the common semantic task.

This is intentionally an **observed annotation inventory**, not a harmonized
label table.

The notebook highlights:
- recordings with no annotations;
- common annotation codes;
- run-level annotation profiles.

In [15]:
# ============================================================
# CELL 13 — OBSERVED LABEL / ANNOTATION OVERVIEW
# ============================================================

if len(annotation_df):
    annotation_profile = (
        annotation_df.groupby(
            ["dataset", "run", "annotation"],
            dropna=False
        )["count"]
        .sum()
        .reset_index()
        .sort_values(
            ["dataset", "run", "count"],
            ascending=[True, True, False]
        )
    )

    no_annotation_recordings = raw_audit_df[
        raw_audit_df["annotation_count"].fillna(0) == 0
    ][
        [
            "dataset",
            "subject",
            "run",
            "filename",
            "file_readable",
        ]
    ]

    print("=" * 78)
    print("OBSERVED ANNOTATION PROFILE")
    print("=" * 78)

    print("\nTop observed annotations:")
    display(
        annotation_df
        .groupby(["dataset", "annotation"])["count"]
        .sum()
        .reset_index()
        .sort_values(
            ["dataset", "count"],
            ascending=[True, False]
        )
        .head(100)
    )

    print("\nRun-level annotation profile:")
    display(annotation_profile.head(150))

    print("\nReadable recordings with no annotations:",
          len(no_annotation_recordings))
    if len(no_annotation_recordings):
        display(no_annotation_recordings.head(100))
else:
    print("No annotations were discovered.")

OBSERVED ANNOTATION PROFILE

Top observed annotations:


,dataset,annotation,count
5,BCI-IV-2a,768,5184
10,BCI-IV-2a,783,2592
6,BCI-IV-2a,769,648
7,BCI-IV-2a,770,648
8,BCI-IV-2a,771,648
9,BCI-IV-2a,772,648
0,BCI-IV-2a,1023,488
4,BCI-IV-2a,32766,160
1,BCI-IV-2a,1072,18
2,BCI-IV-2a,276,17



Run-level annotation profile:


,dataset,run,annotation,count
5,BCI-IV-2a,E,768,2592
6,BCI-IV-2a,E,783,2592
0,BCI-IV-2a,E,1023,224
4,BCI-IV-2a,E,32766,81
1,BCI-IV-2a,E,1072,9
2,BCI-IV-2a,E,276,9
3,BCI-IV-2a,E,277,9
12,BCI-IV-2a,T,768,2592
13,BCI-IV-2a,T,769,648
14,BCI-IV-2a,T,770,648



Readable recordings with no annotations: 0


## Cell 14 — Dataset audit tables

This cell produces compact tables for:
- dataset summary
- subject summary
- run summary
- event/annotation summary
- quality summary

These tables are intended to be saved as CSV for the research record.

In [16]:
# ============================================================
# CELL 14 — AUDIT TABLE GENERATION
# ============================================================

dataset_summary_rows = []

for dataset_name in ["BCI-IV-2a", "EEGMMIDB"]:
    sub_files = files_df[files_df["dataset"] == dataset_name]
    sub_raw = raw_audit_df[raw_audit_df["dataset"] == dataset_name]

    dataset_summary_rows.append({
        "dataset": dataset_name,
        "discovered_files": len(sub_files),
        "raw_eeg_files": len(sub_raw),
        "readable_raw_eeg_files": int(
            sub_raw["file_readable"].fillna(False).sum()
        ) if len(sub_raw) else 0,
        "unique_subjects_parsed": (
            sub_raw.loc[
                sub_raw["subject"].notna()
                & (sub_raw["subject"] != "UNKNOWN"),
                "subject"
            ].nunique()
            if len(sub_raw) else 0
        ),
        "unique_runs_parsed": (
            sub_raw.loc[
                sub_raw["run"].notna()
                & (sub_raw["run"] != "UNKNOWN"),
                ["subject", "run"]
            ].drop_duplicates().shape[0]
            if len(sub_raw) else 0
        ),
        "sampling_rates_hz": (
            sorted(
                [float(x) for x in sub_raw["sfreq_hz"].dropna().unique()]
            )
            if len(sub_raw) else []
        ),
        "channel_counts": (
            sorted(
                [int(x) for x in sub_raw["n_channels"].dropna().unique()]
            )
            if len(sub_raw) else []
        ),
        "total_duration_sec": (
            float(sub_raw["duration_sec"].sum())
            if len(sub_raw) else 0.0
        ),
        "files_sampled_with_nan": int(
            sub_raw["sample_nan"].fillna(False).sum()
        ) if len(sub_raw) else 0,
        "files_sampled_with_inf": int(
            sub_raw["sample_inf"].fillna(False).sum()
        ) if len(sub_raw) else 0,
        "unreadable_files": int(
            (~sub_raw["file_readable"].fillna(False)).sum()
        ) if len(sub_raw) else 0,
    })

dataset_summary_df = pd.DataFrame(dataset_summary_rows)

subject_summary_df = (
    manifest_df.groupby(
        ["dataset", "subject"],
        dropna=False
    )
    .agg(
        recordings=("recording_id", "nunique"),
        total_duration_sec=("duration_sec", "sum"),
        readable_recordings=("file_readable", "sum"),
        annotation_count=("annotation_count", "sum"),
    )
    .reset_index()
    if len(manifest_df)
    else pd.DataFrame()
)

run_summary_df = (
    manifest_df[
        [
            "dataset",
            "subject",
            "session",
            "run",
            "recording_id",
            "filename",
            "sfreq_hz",
            "n_channels",
            "duration_sec",
            "annotation_count",
            "file_readable",
        ]
    ].copy()
    if len(manifest_df)
    else pd.DataFrame()
)

quality_summary_df = (
    raw_audit_df[
        [
            "dataset",
            "subject",
            "run",
            "filename",
            "file_readable",
            "sample_checked",
            "sample_nan",
            "sample_inf",
            "sample_max_abs",
            "sample_ptp_median",
            "read_error",
            "sample_error",
        ]
    ].copy()
    if len(raw_audit_df)
    else pd.DataFrame()
)

print("Dataset summary:")
display(dataset_summary_df)

print("\nSubject summary:")
display(subject_summary_df.head(30))

print("\nRun summary:")
display(run_summary_df.head(50))

print("\nQuality summary:")
display(quality_summary_df.head(50))

Dataset summary:


,dataset,discovered_files,raw_eeg_files,readable_raw_eeg_files,unique_subjects_parsed,unique_runs_parsed,sampling_rates_hz,channel_counts,total_duration_sec,files_sampled_with_nan,files_sampled_with_inf,unreadable_files
0,BCI-IV-2a,47,18,18,9,18,[250.0],[25],48186.64,0,0,0
1,EEGMMIDB,1527,1526,1526,109,1526,"[128.0, 160.0]",[64],174724.00,0,0,0



Subject summary:


,dataset,subject,recordings,total_duration_sec,readable_recordings,annotation_count
0,BCI-IV-2a,S01,2,5438.112,2,1198
1,BCI-IV-2a,S02,2,5359.340,2,1199
2,BCI-IV-2a,S03,2,5237.220,2,1209
3,BCI-IV-2a,S04,2,5043.848,2,1258
4,BCI-IV-2a,S05,2,5463.932,2,1214
5,BCI-IV-2a,S06,2,5381.412,2,1318
6,BCI-IV-2a,S07,2,5416.824,2,1204
7,BCI-IV-2a,S08,2,5452.248,2,1217
8,BCI-IV-2a,S09,2,5393.704,2,1251
9,EEGMMIDB,S001,14,1622.000,14,362



Run summary:


,dataset,subject,session,run,recording_id,filename,sfreq_hz,n_channels,duration_sec,annotation_count,file_readable
0,BCI-IV-2a,S01,E,E,A01E,A01E.gdf,250.0,25,2748.000,595,True
1,BCI-IV-2a,S01,T,T,A01T,A01T.gdf,250.0,25,2690.112,603,True
2,BCI-IV-2a,S02,E,E,A02E,A02E.gdf,250.0,25,2650.664,593,True
3,BCI-IV-2a,S02,T,T,A02T,A02T.gdf,250.0,25,2708.676,606,True
4,BCI-IV-2a,S03,E,E,A03E,A03E.gdf,250.0,25,2595.100,603,True
5,BCI-IV-2a,S03,T,T,A03T,A03T.gdf,250.0,25,2642.120,606,True
6,BCI-IV-2a,S04,E,E,A04E,A04E.gdf,250.0,25,2640.188,648,True
7,BCI-IV-2a,S04,T,T,A04T,A04T.gdf,250.0,25,2403.660,610,True
8,BCI-IV-2a,S05,E,E,A05E,A05E.gdf,250.0,25,2719.452,600,True
9,BCI-IV-2a,S05,T,T,A05T,A05T.gdf,250.0,25,2744.480,614,True



Quality summary:


,dataset,subject,run,filename,file_readable,sample_checked,sample_nan,sample_inf,sample_max_abs,sample_ptp_median,read_error,sample_error
0,BCI-IV-2a,S01,E,A01E.gdf,True,True,False,False,0.000136,0.000103,None,None
1,BCI-IV-2a,S01,T,A01T.gdf,True,True,False,False,0.000360,0.000200,None,None
2,BCI-IV-2a,S02,E,A02E.gdf,True,True,False,False,0.000088,0.000105,None,None
3,BCI-IV-2a,S02,T,A02T.gdf,True,True,False,False,0.000107,0.000167,None,None
4,BCI-IV-2a,S03,E,A03E.gdf,True,True,False,False,0.000073,0.000096,None,None
5,BCI-IV-2a,S03,T,A03T.gdf,True,True,False,False,0.000132,0.000070,None,None
6,BCI-IV-2a,S04,E,A04E.gdf,True,True,False,False,0.000122,0.000088,None,None
7,BCI-IV-2a,S04,T,A04T.gdf,True,True,False,False,0.000601,0.000158,None,None
8,BCI-IV-2a,S05,E,A05E.gdf,True,True,False,False,0.000086,0.000086,None,None
9,BCI-IV-2a,S05,T,A05T.gdf,True,True,False,False,0.000108,0.000082,None,None


## Cell 15 — Save the complete Module 2 manifest

All discovered information is saved under the project's manifest directory.

Important files:
- `dataset_file_inventory.csv`
- `raw_eeg_audit.csv`
- `channel_inventory.csv`
- `observed_annotations.csv`
- `dataset_summary.csv`
- `subject_summary.csv`
- `run_summary.csv`
- `quality_summary.csv`
- `duplicate_candidates.csv`
- `module_2_audit_summary.json`

These outputs form the data-audit record used by later modules.

In [17]:
# ============================================================
# CELL 15 — SAVE MANIFESTS / AUDIT OUTPUTS
# ============================================================

save_map = {
    "dataset_file_inventory.csv": files_df,
    "raw_eeg_audit.csv": raw_audit_df,
    "channel_inventory.csv": channel_inventory_df,
    "observed_annotations.csv": annotation_df,
    "dataset_summary.csv": dataset_summary_df,
    "subject_summary.csv": subject_summary_df,
    "run_summary.csv": run_summary_df,
    "quality_summary.csv": quality_summary_df,
    "duplicate_candidates.csv": duplicate_hash_groups,
}

saved_paths = {}

for filename, dataframe in save_map.items():
    path = MANIFEST_ROOT / filename

    if dataframe is None:
        dataframe = pd.DataFrame()

    dataframe.to_csv(path, index=False)
    saved_paths[filename] = str(path)

audit_summary = {
    "module": 2,
    "project": "cross_dataset_subject_independent_mi_eeg",
    "dataset_roots": {
        "BCI-IV-2a": str(BCI2A_ROOT),
        "EEGMMIDB": str(EEGMMIDB_ROOT),
    },
    "discovered_file_count": int(len(files_df)),
    "raw_eeg_file_count": int(len(raw_audit_df)),
    "readable_raw_eeg_file_count": int(
        raw_audit_df["file_readable"].fillna(False).sum()
    ) if len(raw_audit_df) else 0,
    "unique_subjects": (
        {
            dataset: sorted(
                [
                    x for x in sub["subject"].dropna().unique()
                    if str(x) != "UNKNOWN"
                ]
            )
            for dataset, sub in raw_audit_df.groupby("dataset")
        }
        if len(raw_audit_df)
        else {}
    ),
    "saved_outputs": saved_paths,
    "configuration": {
        "full_sha256_enabled": bool(ENABLE_FULL_SHA256),
        "signal_sample_check_enabled": bool(ENABLE_SIGNAL_SAMPLE_CHECK),
        "signal_sample_seconds": float(SIGNAL_SAMPLE_SECONDS),
    },
}

summary_path = MANIFEST_ROOT / "module_2_audit_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=2, default=str)

print("=" * 78)
print("MANIFESTS SAVED")
print("=" * 78)

for filename, path in saved_paths.items():
    print(f"{filename:35s} -> {path}")

print(f"{'module_2_audit_summary.json':35s} -> {summary_path}")

MANIFESTS SAVED
dataset_file_inventory.csv          -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/dataset_file_inventory.csv
raw_eeg_audit.csv                   -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/raw_eeg_audit.csv
channel_inventory.csv               -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/channel_inventory.csv
observed_annotations.csv            -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/observed_annotations.csv
dataset_summary.csv                 -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/dataset_summary.csv
subject_summary.csv                 -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/subject_summary.csv
run_summary.csv                     -> /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/run_summary.csv
quality_summary.csv                 -> /Users/ashokvarmabevara/Project2/cross_dataset_m

## Cell 16 — Module 2 validation gate

The module is PASS only when:
- both configured dataset roots exist;
- discovery found files;
- raw EEG files were identified;
- the files can be audited;
- subject parsing is not completely unknown;
- the audit outputs were successfully written.

A warning is raised when:
- a dataset has unreadable recordings;
- subject/run names are not fully parsed;
- annotations are absent from some files;
- channel/sampling-rate inconsistencies are present.

These are not automatically failures. They become explicit inputs to the next
modules.

In [18]:
# ============================================================
# CELL 16 — MODULE 2 VALIDATION REPORT
# ============================================================

def count_true(series) -> int:
    if series is None:
        return 0
    return int(series.fillna(False).sum())


validation = {}

validation["bci_root_exists"] = BCI2A_ROOT.exists()
validation["physionet_root_exists"] = EEGMMIDB_ROOT.exists()

validation["files_discovered"] = len(files_df) > 0
validation["raw_files_discovered"] = len(raw_audit_df) > 0

validation["raw_audit_created"] = (
    "dataset" in raw_audit_df.columns
    if len(raw_audit_df)
    else False
)

validation["manifest_saved"] = all(
    Path(path).exists() for path in saved_paths.values()
)

validation["audit_summary_saved"] = summary_path.exists()

if len(raw_audit_df):
    parsed_subjects = raw_audit_df[
        raw_audit_df["subject"].notna()
        & (raw_audit_df["subject"] != "UNKNOWN")
    ]

    validation["subject_parsing_available"] = len(parsed_subjects) > 0
    validation["all_raw_files_readable"] = bool(
        raw_audit_df["file_readable"].fillna(False).all()
    )
    validation["all_signal_samples_finite"] = not bool(
        raw_audit_df["sample_nan"].fillna(False).any()
        or raw_audit_df["sample_inf"].fillna(False).any()
    )
else:
    validation["subject_parsing_available"] = False
    validation["all_raw_files_readable"] = False
    validation["all_signal_samples_finite"] = False

critical_checks = [
    validation["bci_root_exists"],
    validation["physionet_root_exists"],
    validation["files_discovered"],
    validation["raw_files_discovered"],
    validation["raw_audit_created"],
    validation["manifest_saved"],
    validation["audit_summary_saved"],
    validation["subject_parsing_available"],
]

module_status = "PASS" if all(critical_checks) else "FAIL"

warnings_list = []

if len(raw_audit_df):
    unreadable = int(
        (~raw_audit_df["file_readable"].fillna(False)).sum()
    )
    if unreadable:
        warnings_list.append(
            f"{unreadable} raw EEG file(s) could not be read."
        )

    if raw_audit_df["sample_nan"].fillna(False).any():
        warnings_list.append("NaN values detected in sampled signal segments.")

    if raw_audit_df["sample_inf"].fillna(False).any():
        warnings_list.append("Inf values detected in sampled signal segments.")

    if (
        raw_audit_df["parse_status"]
        .fillna("unknown")
        .eq("unknown")
        .any()
    ):
        warnings_list.append("Some recording identities could not be parsed.")

    if (
        raw_audit_df["annotation_count"]
        .fillna(0)
        .eq(0)
        .any()
    ):
        warnings_list.append(
            "Some readable recordings have no annotations."
        )

    if len(raw_audit_df["sfreq_hz"].dropna().unique()) > 1:
        warnings_list.append(
            "Multiple sampling rates were observed. This is expected across "
            "the two datasets but must be handled explicitly later."
        )

    if len(
        raw_audit_df[
            "channel_signature"
        ].dropna().unique()
    ) > 1:
        warnings_list.append(
            "Multiple channel-set patterns were observed. Channel harmonization "
            "must be verified before training."
        )

print("\n" + "=" * 78)
print("MODULE VALIDATION REPORT — MODULE 2")
print("=" * 78)

for key, value in validation.items():
    print(f"{key:38s}: {value}")

print("\nStatus:", module_status)

if warnings_list:
    print("\nWarnings:")
    for item in warnings_list:
        print("  -", item)

if module_status == "PASS" and warnings_list:
    print("\nFINAL MODULE STATUS: PASS WITH WARNING")
    print("The audit completed, but the listed conditions must be reviewed")
    print("before locking the label/channel protocol.")
elif module_status == "PASS":
    print("\nFINAL MODULE STATUS: PASS")
    print("Dataset discovery and audit completed successfully.")
else:
    print("\nFINAL MODULE STATUS: FAIL")
    print("Do NOT proceed to Module 3 until the critical checks are fixed.")


MODULE VALIDATION REPORT — MODULE 2
bci_root_exists                       : True
physionet_root_exists                 : True
files_discovered                      : True
raw_files_discovered                  : True
raw_audit_created                     : True
manifest_saved                        : True
audit_summary_saved                   : True
subject_parsing_available             : True
all_raw_files_readable                : True
all_signal_samples_finite             : True

Status: PASS

Warnings:
  - Multiple sampling rates were observed. This is expected across the two datasets but must be handled explicitly later.
  - Multiple channel-set patterns were observed. Channel harmonization must be verified before training.

FINAL MODULE STATUS: PASS WITH WARNING
The audit completed, but the listed conditions must be reviewed
before locking the label/channel protocol.


## Cell 17 — Research handoff summary

This cell prints the information that must be reviewed manually before Module 3.

Module 3 will use these observations to decide the valid common label space.

In [19]:
# ============================================================
# CELL 17 — RESEARCH HANDOFF SUMMARY
# ============================================================

print("=" * 78)
print("MODULE 2 RESEARCH HANDOFF")
print("=" * 78)

if len(dataset_summary_df):
    display(dataset_summary_df)

if len(subject_summary_df):
    print("\nSubject counts:")
    print(
        subject_summary_df.groupby("dataset")["subject"]
        .nunique()
        .to_string()
    )

print("\nObserved annotation labels by dataset:")

if len(annotation_df):
    for dataset_name, sub in annotation_df.groupby("dataset"):
        labels = (
            sub.groupby("annotation")["count"]
            .sum()
            .sort_values(ascending=False)
        )
        print(f"\n{dataset_name}")
        print(labels.to_string())
else:
    print("No annotations available.")

print("\nCommon channel names (normalized intersection):")

channel_sets = {}

if len(channel_inventory_df):
    for dataset_name, sub in channel_inventory_df.groupby("dataset"):
        channel_sets[dataset_name] = set(
            sub["channel_name_normalized"].dropna().unique()
        )

    if "BCI-IV-2a" in channel_sets and "EEGMMIDB" in channel_sets:
        intersection = sorted(
            channel_sets["BCI-IV-2a"]
            & channel_sets["EEGMMIDB"]
        )
        print("Exact normalized-name intersection count:", len(intersection))
        print(intersection)
    else:
        print("Both dataset channel inventories are not available.")
else:
    print("No channel inventory available.")

print("\nIMPORTANT:")
print("The apparent channel intersection is NOT yet the final common montage.")
print("Module 4 must verify electrode identity, montage coordinates, and")
print("sensorimotor coverage before freezing the channel set.")

MODULE 2 RESEARCH HANDOFF


,dataset,discovered_files,raw_eeg_files,readable_raw_eeg_files,unique_subjects_parsed,unique_runs_parsed,sampling_rates_hz,channel_counts,total_duration_sec,files_sampled_with_nan,files_sampled_with_inf,unreadable_files
0,BCI-IV-2a,47,18,18,9,18,[250.0],[25],48186.64,0,0,0
1,EEGMMIDB,1527,1526,1526,109,1526,"[128.0, 160.0]",[64],174724.00,0,0,0



Subject counts:
dataset
BCI-IV-2a      9
EEGMMIDB     109

Observed annotation labels by dataset:

BCI-IV-2a
annotation
768      5184
783      2592
769       648
770       648
771       648
772       648
1023      488
32766     160
1072       18
276        17
277        17

EEGMMIDB
annotation
T0    19893
T1     9858
T2     9818

Common channel names (normalized intersection):
Exact normalized-name intersection count: 0
[]

IMPORTANT:
The apparent channel intersection is NOT yet the final common montage.
Module 4 must verify electrode identity, montage coordinates, and
sensorimotor coverage before freezing the channel set.


# MODULE 2 STOP CONDITION

Do not proceed automatically.

Before Module 3, manually inspect:

1. the discovered subject lists;
2. the observed run/session structure;
3. annotation distributions;
4. unreadable/corrupted recordings;
5. channel-name patterns;
6. sampling rates;
7. missing runs;
8. duplicate candidates.

The next module will make the **semantic label decision** only after this audit.

**Do not manually rename or delete data files based on these tables.**